# CyberFin Fusion - Detection Engine Demo
Interactive notebook for testing mule ring detection and risk scoring

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from enhanced_detection import EnhancedDetector

print("✅ Imports successful")

## 1. Load Data

In [ ]:
cyber_df = pd.read_csv('cyber_events.csv')
txn_df = pd.read_csv('transactions.csv')

print(f"Cyber Events: {len(cyber_df):,}")
print(f"Transactions: {len(txn_df):,}")

cyber_df.head()

## 2. Build Graph

In [ ]:
G = nx.Graph()

# Add cyber events
for _, row in cyber_df.iterrows():
    G.add_node(row['account_id'], type='account')
    G.add_node(f"IP_{row['ip']}", type='ip')
    G.add_node(f"DEV_{row['device']}", type='device')
    G.add_edge(row['account_id'], f"IP_{row['ip']}")
    G.add_edge(row['account_id'], f"DEV_{row['device']}")

# Add transactions
for _, row in txn_df.iterrows():
    G.add_edge(row['account_id'], row['beneficiary'], amount=row['amount'])

print(f"Graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")

## 3. Initialize Detector

In [ ]:
detector = EnhancedDetector(G)
print("✅ Detector initialized")

## 4. Find Mule Rings

In [ ]:
rings = detector.find_mule_rings()

print(f"🔍 Found {len(rings)} mule rings\n")

# Show top 5 rings
for i, ring in enumerate(rings[:5]):
    print(f"Ring {i+1}:")
    print(f"  Size: {ring['size']} accounts")
    print(f"  Risk: {ring['ring_risk_score']:.1f}/100")
    print(f"  Beneficiaries: {', '.join(ring['shared_beneficiaries'][:3])}")
    print()

## 5. Visualize Largest Ring

In [ ]:
if rings:
    largest_ring = rings[0]
    ring_accounts = largest_ring['accounts'][:20]  # First 20 for visualization
    
    # Create subgraph
    subgraph_nodes = set(ring_accounts)
    for acc in ring_accounts:
        subgraph_nodes.update(G.neighbors(acc))
    
    subgraph = G.subgraph(subgraph_nodes)
    
    # Visualize
    plt.figure(figsize=(15, 10))
    pos = nx.spring_layout(subgraph, k=0.5, iterations=50)
    
    # Color nodes by type
    node_colors = []
    for node in subgraph.nodes():
        if node.startswith('ACC_'):
            node_colors.append('red')
        elif node.startswith('BEN_'):
            node_colors.append('orange')
        else:
            node_colors.append('lightblue')
    
    nx.draw(subgraph, pos, node_color=node_colors, 
            with_labels=True, node_size=500, font_size=8)
    
    plt.title(f"Mule Ring Visualization - {largest_ring['size']} accounts")
    plt.show()

## 6. Calculate Risk Scores

In [ ]:
# Test specific accounts
test_accounts = ['ACC_002747', 'ACC_004611', 'ACC_000815']

for acc in test_accounts:
    risk = detector.calculate_risk(acc)
    print(f"{acc}: Risk = {risk}/100")

## 7. Get High-Risk Accounts

In [ ]:
high_risk = detector.get_high_risk_accounts(threshold=70, limit=20)

print(f"🚨 Found {len(high_risk)} critical risk accounts\n")

# Create DataFrame
risk_df = pd.DataFrame(high_risk, columns=['Account', 'Risk Score'])
risk_df

## 8. Real-Time Anomaly Detection

In [ ]:
# Simulate real-time event
test_account = 'ACC_002747'
is_anomaly, anomalies = detector.detect_anomalies_realtime(
    test_account,
    'transaction',
    {'amount': 48000, 'location': 'Romania'}
)

print(f"Account: {test_account}")
print(f"Anomaly Detected: {is_anomaly}")
print(f"\nAnomalies:")
for anomaly_type, severity in anomalies:
    print(f"  • {anomaly_type}: {severity.upper()}")

## 9. Generate Alert

In [ ]:
risk = detector.calculate_risk(test_account)
alert = detector.generate_alert(test_account, risk, anomalies)

import json
print(json.dumps(alert, indent=2))

## 10. System Statistics

In [ ]:
stats = detector.get_statistics()

print("📊 Detection System Statistics\n")
for key, value in stats.items():
    print(f"{key.replace('_', ' ').title()}: {value:,}")

## 11. Risk Distribution

In [ ]:
# Calculate risk for all accounts
accounts = [n for n in G.nodes() if n.startswith('ACC_')]
risks = [detector.calculate_risk(acc) for acc in accounts[:1000]]  # Sample 1000

plt.figure(figsize=(12, 6))
plt.hist(risks, bins=20, color='red', alpha=0.7, edgecolor='black')
plt.xlabel('Risk Score')
plt.ylabel('Number of Accounts')
plt.title('Risk Score Distribution')
plt.axvline(x=50, color='orange', linestyle='--', label='High Risk Threshold')
plt.axvline(x=70, color='red', linestyle='--', label='Critical Risk Threshold')
plt.legend()
plt.grid(alpha=0.3)
plt.show()